In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Generalizability Evaluation for InterpDetect

This notebook evaluates the generalizability of findings in the InterpDetect repository.

## Evaluation Checklist:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data  
- **GT3**: Method / Specificity Generalizability

## Constraints:
- Up to 3 trial examples for GT1 and GT2
- Up to 3 similar tasks for GT3
- Models used must not appear in original paper
- Data used must not appear in original dataset

In [2]:
# Explore the InterpDetect repository structure
repo_path = '/net/scratch2/smallyan/InterpDetect_eval'

# List top-level contents
for item in sorted(os.listdir(repo_path)):
    item_path = os.path.join(repo_path, item)
    if os.path.isdir(item_path):
        print(f"📁 {item}/")
    else:
        print(f"📄 {item}")

📁 .git/
📄 .gitignore
📄 CodeWalkthrough.md
📄 LICENSE
📁 datasets/
📄 documentation.pdf
📁 evaluation/
📄 plan.md
📄 requirements.txt
📁 scripts/
📁 trained_models/


In [3]:
# Read the plan.md to understand the research
with open(os.path.join(repo_path, 'plan.md'), 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
Develop a mechanistic interpretability-based hallucination detection method for Retrieval-Augmented Generation (RAG) systems by computing External Context Scores (ECS) across layers and attention heads and Parametric Knowledge Scores (PKS) across layers (FFN), training regression-based classifiers on these signals, and demonstrating generalization from a small proxy model (Qwen3-0.6b) to larger production models (GPT-4.1-mini).

## Hypothesis
1. RAG hallucinations correlate with:  later-layer FFN modules disproportionately inject parametric knowledge into the residual stream while attention heads fail to adequately exploit external context.
2. External Context Score (ECS) and Parametric Knowledge Score (PKS) are correlated with hallucination occurrence and can serve as predictive features for hallucination detection.
3. Mechanistic signals extracted from a small proxy model (0.6b parameters) can generalize to detect hallucinations in responses from larger production

In [4]:
# Read CodeWalkthrough for more details on implementation
with open(os.path.join(repo_path, 'CodeWalkthrough.md'), 'r') as f:
    walkthrough_content = f.read()
print(walkthrough_content)

# InterpDetect

**InterpDetect: Interpretable Signals for Detecting Hallucinations in Retrieval-Augmented Generation**

[![Paper](https://img.shields.io/badge/Paper-OpenReview-blue)](https://openreview.net/pdf?id=TZzBKwHLwF)
[![License](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)

A comprehensive framework for detecting and analyzing hallucinations in Retrieval-Augmented Generation (RAG) systems using interpretability techniques and chunk-level analysis on the RAGBench/FinQA dataset.

## Overview

This project implements a novel approach to hallucination detection by leveraging interpretability methods. The framework consists of three main parts:

1. **Preprocessing Pipeline** - Convert raw datasets to labeled data with hallucination spans
2. **Training & Prediction** - Train classifiers or use pre-trained models for hallucination detection
3. **Baseline Comparisons** - Evaluate against multiple baseline methods (RAGAS, TruLens, RefChecker, GPT-based, etc.)

The frame

In [5]:
# Explore the datasets directory
datasets_path = os.path.join(repo_path, 'datasets')
for item in sorted(os.listdir(datasets_path)):
    item_path = os.path.join(datasets_path, item)
    if os.path.isdir(item_path):
        print(f"📁 {item}/")
        for sub_item in sorted(os.listdir(item_path)):
            print(f"    📄 {sub_item}")
    else:
        print(f"📄 {item}")

📄 OV_copying_score.json
📁 test/
    📄 test_w_chunk_score_gpt41mini.json
    📄 test_w_chunk_score_qwen06b.json
📁 train/
    📄 train3000_w_chunk_score_part0.json
    📄 train3000_w_chunk_score_part1.json
    📄 train3000_w_chunk_score_part10.json
    📄 train3000_w_chunk_score_part11.json
    📄 train3000_w_chunk_score_part12.json
    📄 train3000_w_chunk_score_part13.json
    📄 train3000_w_chunk_score_part14.json
    📄 train3000_w_chunk_score_part15.json
    📄 train3000_w_chunk_score_part16.json
    📄 train3000_w_chunk_score_part17.json
    📄 train3000_w_chunk_score_part2.json
    📄 train3000_w_chunk_score_part3.json
    📄 train3000_w_chunk_score_part4.json
    📄 train3000_w_chunk_score_part5.json
    📄 train3000_w_chunk_score_part6.json
    📄 train3000_w_chunk_score_part7.json
    📄 train3000_w_chunk_score_part8.json
    📄 train3000_w_chunk_score_part9.json


In [6]:
# Explore the trained_models directory
models_path = os.path.join(repo_path, 'trained_models')
for item in sorted(os.listdir(models_path)):
    print(f"📄 {item}")

📄 model_LR_3000.pickle
📄 model_RandomForest_3000.pickle
📄 model_SVC_3000.pickle
📄 model_XGBoost_3000.pickle


In [7]:
# Explore the scripts directory
scripts_path = os.path.join(repo_path, 'scripts')
for item in sorted(os.listdir(scripts_path)):
    item_path = os.path.join(scripts_path, item)
    if os.path.isdir(item_path):
        print(f"📁 {item}/")
        for sub_item in sorted(os.listdir(item_path)):
            print(f"    📄 {sub_item}")
    else:
        print(f"📄 {item}")

📄 .DS_Store
📁 baseline/
    📄 requirements.txt
    📄 run_gpt.py
    📄 run_groq.py
    📄 run_hf.py
    📄 run_ragas.py
    📄 run_refchecker.py
    📄 run_trulens.py
📄 classifier.py
📄 compute_scores.py
📁 plots/
    📄 plot_correlation.ipynb
📄 predict.py
📁 preprocess/
    📄 README.md
    📄 datasets
    📄 filter.py
    📄 generate_labels.py
    📄 generate_response_gpt.py
    📄 generate_response_hf.py
    📄 helper.py
    📄 preprocess.py


In [8]:
# Read the compute_scores.py to understand how ECS and PKS are computed
with open(os.path.join(scripts_path, 'compute_scores.py'), 'r') as f:
    compute_scores_content = f.read()
print(compute_scores_content)

# %%
#!pip install transformer_lens

import torch
from transformers import AutoTokenizer
from transformer_lens import HookedTransformer
import json
from torch.nn import functional as F
from typing import Dict, List, Tuple
import pdb
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import argparse
import sys
import os
import gc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr

def load_examples(file_path):
    """Load examples from JSONL file"""
    print(f"Loading examples from {file_path}...")
    
    try:
        examples = []
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                examples.append(data)
        
        print(f"Loaded {len(examples)} examples")
        return examples
    except Exception as e:
        print(f"Error loading examples: {e}")
        sys.exit(1)

def setup_models(model_name, hf_

In [9]:
# Load and examine the test data to understand the structure
import json

test_data_path = os.path.join(repo_path, 'datasets/test/test_w_chunk_score_qwen06b.json')
with open(test_data_path, 'r') as f:
    test_data = json.load(f)
    
print(f"Number of test examples: {len(test_data)}")
print(f"\nKeys in each example: {test_data[0].keys()}")
print(f"\nFirst example structure:")
for key, val in test_data[0].items():
    if isinstance(val, str):
        print(f"  {key}: {val[:100]}..." if len(val) > 100 else f"  {key}: {val}")
    elif isinstance(val, list) and len(val) > 0:
        print(f"  {key}: list of {len(val)} items")
    else:
        print(f"  {key}: {val}")

Number of test examples: 256

Keys in each example: dict_keys(['id', 'question', 'documents', 'documents_sentences', 'prompt', 'prompt_spans', 'num_tokens', 'response', 'response_spans', 'labels', 'hallucinated_llama-4-maverick-17b-128e-instruct', 'hallucinated_gpt-oss-120b', 'labels_llama', 'labels_gpt', 'scores'])

First example structure:
  id: finqa_6345
  question: what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?
  documents: list of 3 items
  documents_sentences: list of 3 items
  prompt: Given the context, please answer the question based on the provided information from the context. In...
  prompt_spans: list of 8 items
  num_tokens: 639
  response: The rate of return in Cadence Design Systems Inc. for an investment from 2010 to 2011 can be calcula...
  response_spans: list of 5 items
  labels: list of 13 items
  hallucinated_llama-4-maverick-17b-128e-instruct: No
- The response incorrectly calculates the rate of return. The rate of